# Gravità: un solo neurone

Stessa formula, **F = m × g**, ma con un solo neurone lineare: F = w × m + b.
Se impara bene, il peso w deve avvicinarsi a 9.8.

Prova a cambiare `learning_rate`: con 0.000001 arriva a 9.8; con 0.00001 il passo è troppo grande e il modello esplode.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers, models
from sklearn.preprocessing import StandardScaler

# Constants
g = 9.8  # gravity on Earth in m/s^2

# Generate synthetic data
def generate_data(num_samples=1000):
    m = np.random.uniform(1, 1000, num_samples)  # Mass in kg
    F = m * g  # Force (weight) in Newtons
    return m, F

# Generate training and testing data
m_train, F_train = generate_data(800)
m_test, F_test = generate_data(200)

# Reshape the data
m_train = m_train.reshape(-1, 1)  # TensorFlow expects input as (batch_size, num_features)
m_test = m_test.reshape(-1, 1)

'''
scaler = StandardScaler() #Scale the input features to a similar range to help the model learn more efficiently.
m_train = scaler.fit_transform(m_train)
m_test = scaler.transform(m_test)
'''
#Build the model
'''
model = models.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'), # Additional layer
    layers.Dense(1, activation='linear')  # Linear activation for output
])
'''
model = models.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(1, activation='linear')
])
'''
# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')
'''

# Compile the model with a lower learning rate and a different optimizer
#optimizer = tf.keras.optimizers.SGD(learning_rate=0.00001)

# con learning_rate=0.0001 e clipvalue=1.0 il peso si fermava a circa 1,9: passo troppo piccolo
# prova 0.00001: il passo è troppo grande e il modello esplode (loss = nan)
optimizer = tf.keras.optimizers.SGD(learning_rate=0.000001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

'''
# Train the model
model.fit(m_train, F_train, epochs=100, batch_size=32)
'''

model.fit(m_train, F_train, epochs=20, batch_size=8)

'''
# Compile the model with a lower learning rate
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mean_squared_error')

# Introduce early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

# Train the model with validation data and early stopping
model.fit(m_train, F_train, epochs=1000, batch_size=32,
          validation_data=(m_test, F_test), callbacks=[early_stopping])
'''

# Evaluate the model
loss = model.evaluate(m_test, F_test)
print(f"Test Loss: {loss}")

# Make predictions
predictions = model.predict(m_test)

In [ ]:
# Compare predictions to actual values
import matplotlib.pyplot as plt

# Make predictions on the ORIGINAL, unscaled test data
predictions_unscaled = model.predict(m_test)

plt.figure(figsize=(10, 6))
plt.scatter(F_test, predictions_unscaled.flatten(), alpha=0.5, color='#1F3A5F')
plt.plot([min(F_test), max(F_test)], [min(F_test), max(F_test)], color='#C25A00', linestyle='--')
plt.xlabel("Actual Force (N)")
plt.ylabel("Predicted Force (N)")
plt.title("Actual vs Predicted Force (Weight)")
plt.show()

In [ ]:
# Extract the learned weights and biases
weights, biases = model.layers[0].get_weights()

# Extract the learned gravitational constant (g) and bias
g_learned = weights[0][0]
b = biases[0]

# Print the learned equation
print(f"Learned equation: F = {g_learned} * m + {b}")

Learned equation: F = 9.800000190734863 * m + 0.0


Controprova: ora **scriviamo noi** w = 9.8 e b = 0, e confrontiamo con i valori veri.
Questi numeri non li ha imparati il modello: servono solo a vedere la risposta perfetta.

In [ ]:
import pandas as pd

# Set weights directly (assuming a single Dense layer)
expected_g = 9.8
expected_bias = 0.0
model.layers[0].set_weights([np.array([[expected_g]]), np.array([expected_bias])])

# Make predictions using the manually set weights
predictions = model.predict(m_test)

# Compare predictions to actual values
comparison = pd.DataFrame({'Mass': m_test.flatten(), 'Actual Force': F_test, 'Predicted Force': predictions.flatten()})
print(comparison)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
           Mass  Actual Force  Predicted Force
0    459.628970   4504.363910      4504.363770
1    111.330006   1091.034062      1091.034058
2    207.686873   2035.331357      2035.331421
3    829.367903   8127.805454      8127.805664
4    367.064657   3597.233636      3597.233887
..          ...           ...              ...
195  730.952514   7163.334635      7163.334961
196  178.914524   1753.362336      1753.362305
197  239.503330   2347.132633      2347.132568
198  265.713536   2603.992656      2603.992676
199  621.600195   6091.681914      6091.682129

[200 rows x 3 columns]
